In [ ]:
%matplotlib

import sys,os
import time, datetime, dateutil
import re, glob, json

import numpy as np
import pandas as pd
import itertools

pd.set_option('display.max_rows', 999)
pd.set_option('display.min_rows', 100)
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = '{:,.2f}'.format

import scipy
import logging
from scipy import stats

import sklearn
from sklearn.linear_model import LinearRegression
from skimage import filters
from scipy.interpolate import interp1d
from scipy.ndimage import distance_transform_edt
from statsmodels.regression.mixed_linear_model import MixedLM
from sklearn.preprocessing import MinMaxScaler
from skimage import filters
from typing import Dict, Tuple, List
from collections import Counter, defaultdict

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.pyplot import *

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import torch
import torch.nn as nn
import torch.profiler
import torch.nn.functional as tf
torch.set_default_dtype(torch.float32)

Using matplotlib backend: <object object at 0x78ef03a730e0>


SOM Training (with Frequency-Based Learning)

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using MPS (Apple Silicon)")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

FTYPE = torch.float32
ITYPE = torch.int64
torch.set_default_dtype(torch.float32)
_ZERO  = 0.0
_ONE   = 1.0
_TINY  = 1e-10
_LOG2  = float(np.log(2.0))


data_dir = ""   # <- set directory as needed

word_info = pd.read_csv(data_dir + "word_info_small_2025_2_11.csv",  index_col=0)
eng_data = pd.read_csv(data_dir + "english_small_2025_2_11.csv",    index_col=0)
spa_data = pd.read_csv(data_dir + "spanish_small_2025_2_11.csv",    index_col=0)
sem_data = pd.read_csv(data_dir + "semantics_small_2025_2_11.csv",  index_col=0)


# Normalize word frequencies
word_info.loc['living thing', 'spa_freq'] = word_info.loc['living thing', 'eng_freq']
sem_data["word_freq"] = word_info["eng_freq"]
sem_data.loc[:, "normalized_word_freq"] =  np.log1p(sem_data["word_freq"])
eng_data["word_freq"] = word_info["eng_freq"]
eng_data.loc[:, "normalized_word_freq"] =  np.log1p(eng_data["word_freq"])
spa_data["word_freq"] = word_info["spa_freq"]
spa_data.loc[:, "normalized_word_freq"] =  np.log1p(spa_data["word_freq"])

eng_freq = eng_data.loc[eng_data.index, "normalized_word_freq"]
spa_freq = spa_data.loc[spa_data.index, "normalized_word_freq"]
sem_freq = sem_data.loc[sem_data.index, "normalized_word_freq"]


def build_lateral_weights(x, y, sigma=None, mode="constant", truncate=5.0):
    lw = np.zeros((x, y, x, y), dtype=np.float32)
    for xi in range(x):
        for yi in range(y):
            one = np.zeros((x, y), dtype=np.float32)
            one[xi, yi] = 1.0
            lw[xi, yi] = (
                filters.gaussian(one, sigma=sigma, mode=mode, truncate=truncate)
                if sigma is not None
                else one
            ).astype(np.float32)
    return lw


def normalize_lateral_weights(lat_ex_, how="in", exp=0.5):
    shape = lat_ex_.shape
    if how == "in":
        wsum = torch.pow(torch.sum(lat_ex_, dim=(2, 3)), exp)
        lat_ex_ = lat_ex_ / (wsum.reshape(shape[0], shape[1], 1, 1) + _TINY)
    elif how == "out":
        wsum = torch.pow(torch.sum(lat_ex_, dim=(0, 1)), exp)
        lat_ex_ = lat_ex_ / (wsum.reshape(1, 1, shape[2], shape[3]) + _TINY)
    else:
        raise ValueError("how must be 'in' or 'out'")
    return lat_ex_


def median_dist(df, sqrt=False):
    diff = np.square(
        df.values.reshape([len(df), -1, 1]) -
        df.values.reshape([len(df), -1, 1]).T
    )
    dist = pd.DataFrame(
        index=df.index, columns=df.index,
        data=(np.sqrt(np.sum(diff, axis=1)) if sqrt else np.sum(diff, axis=1))
    ).stack()
    return dist[
        dist.index.get_level_values(0) != dist.index.get_level_values(1)
    ].median()


def make_decay(initial=1.0, halflife=100):
    return lambda epoch: initial * 0.5 ** (epoch / float(halflife))


def make_seeded_weights(data, x, y, noise_scale=0.01, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    values = data.values.astype(np.float32)
    n_inputs, dim = values.shape
    n_units = x * y

    if n_units < n_inputs:
        raise ValueError(
            f"Map has only {n_units} units but there are {n_inputs} inputs."
        )

    W = np.empty((n_units, dim), dtype=np.float32)
    data_std = np.std(values, axis=0, keepdims=True).astype(np.float32)
    data_std[data_std == 0] = 1.0

    perm_inputs  = rng.permutation(n_inputs)
    chosen_units = rng.choice(n_units, size=n_inputs, replace=False)
    W[:] = np.nan

    for input_idx, unit_idx in zip(perm_inputs, chosen_units):
        jitter = noise_scale * data_std[0] * rng.standard_normal(dim).astype(np.float32)
        W[unit_idx] = values[input_idx] + jitter

    remaining_units = np.where(np.isnan(W).any(axis=1))[0]
    source_inputs   = rng.integers(0, n_inputs, size=len(remaining_units))
    for unit_idx, input_idx in zip(remaining_units, source_inputs):
        jitter = noise_scale * data_std[0] * rng.standard_normal(dim).astype(np.float32)
        W[unit_idx] = values[input_idx] + jitter

    return W.reshape(x, y, dim)


def compute_expected_cells(freq, x, y):
    freq = np.asarray(freq, dtype=np.float64)
    probs      = freq / freq.sum()
    densities  = probs ** (2.0 / 3.0)
    return (densities / densities.sum() * (x * y)).astype(np.float32)


def compute_isolation(data):
    values = data.values.astype(np.float32)
    n = len(values)
    isolation = np.empty(n, dtype=np.float32)
    for i in range(n):
        dists = np.sqrt(((values - values[i]) ** 2).sum(axis=1))
        isolation[i] = np.sort(dists)[5]
    return isolation


def blend_sampling_weights(freq, isolation, isolation_weight=0.3):
    freq      = np.asarray(freq,      dtype=np.float64)
    isolation = np.asarray(isolation, dtype=np.float64)
    freq_norm      = freq      / freq.sum()
    isolation_norm = isolation / isolation.sum()
    blended = (1.0 - isolation_weight) * freq_norm + isolation_weight * isolation_norm
    return (blended / blended.sum()).astype(np.float64)



@torch.compile(backend='inductor')
@torch.no_grad
def _compiled_next_act(act_, lat_, decay, tiny):
    lateral_drive = torch.sum(act_.reshape(act_.shape[0], act_.shape[1], 1, 1) * lat_, dim=(0, 1))
    next_ = torch.nn.functional.relu(act_ * decay + lateral_drive)
    next_ = next_ * (1.0 / (next_.sum() + tiny))
    delta_ = torch.sqrt(torch.sum((next_ - act_) ** 2))
    return next_, delta_


@torch.compile(backend='inductor')
@torch.no_grad
def _compiled_update_labels(data_, W_):
    raw_diff_ = data_.view(data_.shape[0], 1, 1, data_.shape[1]) - W_.view(1, W_.shape[0], W_.shape[1], W_.shape[2])
    dist2_    = torch.sum(raw_diff_.pow(2), dim=3)
    return torch.argmin(dist2_, dim=0)


@torch.compile(backend='inductor')
@torch.no_grad
def _compiled_set_lateral(x, y, sigma_, lat_strength, lat_in_val, device, ftype):
    X      = torch.arange(0.0, x, 1.0, dtype=ftype, device=device)
    Y      = torch.arange(0.0, y, 1.0, dtype=ftype, device=device)
    xx, yy = torch.meshgrid(X, Y, indexing="ij")
    dx     = xx.reshape(x, y, 1, 1) - xx.reshape(1, 1, x, y)
    dy     = yy.reshape(x, y, 1, 1) - yy.reshape(1, 1, x, y)
    d2     = torch.square(dx) + torch.square(dy)
    return torch.exp(-d2 / (2.0 * sigma_ * sigma_)) / (sigma_ * sigma_ * torch.pi * 2.0)


class LatSom(nn.Module):
    def __init__(self, data, x=12, y=12, W=None):
        super().__init__()
        self.device = DEVICE

        if not isinstance(data, pd.DataFrame):
            raise TypeError("data must be a pandas DataFrame")

        with torch.inference_mode():
            self.data  = data
            self.data_ = torch.tensor(data.values, dtype=FTYPE, device=self.device)

            self.max_bits    = np.log(len(self.data)) / np.log(2.0)
            self.median_dist = float(median_dist(self.data))

            if W is not None:
                if W.ndim != 3:
                    raise ValueError("W must have shape (x, y, dim)")
                self.x, self.y, self.dim = W.shape
            else:
                self.x, self.y, self.dim = x, y, data.shape[1]
                W = (
                    0.01 * np.random.standard_normal((self.x, self.y, self.dim))
                    * np.std(self.data.values,  axis=0).reshape(1, 1, -1)
                    + np.mean(self.data.values, axis=0).reshape(1, 1, -1)
                ).astype(np.float32)

            self.W_      = torch.tensor(W, dtype=FTYPE, device=self.device)
            self.act_    = torch.zeros((self.x, self.y), dtype=FTYPE, device=self.device)
            self.labels_ = torch.zeros((self.x, self.y), dtype=ITYPE, device=self.device)

            self.lat_ex_ = None
            self.lat_in_ = None
            self.lat_    = None


    def get_input_tensor(self, input):
        with torch.inference_mode():
            if isinstance(input, str):
                matches = np.where(self.data.index == input)[0]
                if len(matches) == 0:
                    raise ValueError(f"Input label not found: {input}")
                input = int(matches[0])
            if isinstance(input, (int, np.integer)):
                input = self.data_[input]
            elif isinstance(input, np.ndarray):
                if input.shape != (self.dim,):
                    raise ValueError(f"NumPy input must have shape ({self.dim},)")
                input = torch.tensor(input, dtype=FTYPE, device=self.device)
            elif isinstance(input, torch.Tensor):
                input = input.to(self.device, dtype=FTYPE)
            else:
                raise TypeError("Bad input pattern")
            return input

    def present(self, input, T=1.0, strength=1.0):
        with torch.inference_mode():
            input     = self.get_input_tensor(input)
            raw_diff_ = input.reshape(1, 1, self.dim) - self.W_
            dist2_    = torch.sum(raw_diff_.pow(2), dim=2)
            torch.exp(dist2_ / (-self.median_dist * T + _TINY), out=self.act_)
            self.act_ /= (torch.sum(self.act_) / strength + _TINY)
            winner_   = torch.argmin(dist2_.reshape(self.x * self.y)).to(ITYPE)
            return winner_ // self.y, winner_ % self.y

    def present_batch(self, batch=None):
        with torch.inference_mode():
            input_    = self.data_ if batch is None else self.data_[batch, :]
            raw_diff_ = input_.view(-1, 1, 1, self.dim) - self.W_.view(1, self.x, self.y, self.dim)
            dist2_    = torch.sum(raw_diff_.pow(2), dim=3)
            winner_   = torch.argmin(dist2_.reshape(-1, self.x * self.y), dim=1)
            return winner_ // self.y, winner_ % self.y

    def update_labels(self):
        with torch.inference_mode():
            self.labels_ = _compiled_update_labels(self.data_, self.W_).to(ITYPE)
            return self.labels_

    def territory_sizes(self):
        with torch.inference_mode():
            self.update_labels()
            return torch.bincount(self.labels_.reshape(-1), minlength=len(self.data))

    def get_underrepresented(self, expected_cells, zero_threshold=0, ratio_threshold=0.5):
        with torch.inference_mode():
            counts         = self.territory_sizes().cpu().numpy().astype(np.float32)
            expected       = np.asarray(expected_cells, dtype=np.float32)
            under_mask     = (counts <= zero_threshold) | (counts < expected * ratio_threshold)
            indices        = np.where(under_mask)[0]
            deficit_ratios = np.clip(1.0 - counts[indices] / (expected[indices] + 1e-10), 0.0, 1.0)
            return indices, deficit_ratios, counts

    def get_overrepresented(self, expected_cells, counts, ceiling_factor=2.0):
        with torch.inference_mode():
            expected      = np.asarray(expected_cells, dtype=np.float32)
            over_mask     = counts > expected * ceiling_factor
            over_indices  = np.where(over_mask)[0]
            excess_ratios = np.clip(
                (counts[over_indices] - expected[over_indices]) / (expected[over_indices] + 1e-10),
                0.0, None,
            )
            return over_indices, excess_ratios

    def penalise_overrepresented(
        self, over_indices, excess_ratios, expected_cells, counts,
        alpha, sigma, thresh, steps, decay, beta=0.5, in_sigma=1.5, T=1.0,
    ):
        with torch.inference_mode():
            expected = np.asarray(expected_cells, dtype=np.float32)
            for idx, excess in zip(over_indices, excess_ratios):
                n_excess  = max(1, int(round(counts[idx] - expected[idx])))
                n_retrain = max(1, min(n_excess, int(round(excess))))
                owned     = torch.nonzero(self.labels_ == int(idx), as_tuple=False)
                if owned.shape[0] <= 1:
                    continue
                input_vec     = self.data_[int(idx)]
                owned_weights = self.W_[owned[:, 0], owned[:, 1], :]
                dists         = torch.sum((owned_weights - input_vec) ** 2, dim=1)
                n_retrain     = min(n_retrain, owned.shape[0] - 1)
                _, far_order  = torch.sort(dists, descending=True)
                far_units     = owned[far_order[:n_retrain]]
                for k in range(far_units.shape[0]):
                    ui, uj    = int(far_units[k, 0]), int(far_units[k, 1])
                    unit_w    = self.W_[ui, uj, :]
                    all_dists = torch.sum((self.data_ - unit_w.unsqueeze(0)) ** 2, dim=1)
                    all_dists[int(idx)] = float("inf")
                    nn_idx = int(torch.argmin(all_dists).item())
                    self.train(
                        input=nn_idx, alpha=alpha, delta_threshold=thresh,
                        max_steps=steps, decay=decay, beta=beta, in_sigma=in_sigma, T=T,
                    )

    def next_act(self, decay=0.25, other_input=None):
        with torch.inference_mode():
            next_, delta_ = _compiled_next_act(self.act_, self.lat_, decay, _TINY)
            if other_input is not None:
                self.act_.copy_(next_ + other_input)
            else:
                self.act_.copy_(next_)
            return delta_

    def class_hist(self, T=None, soft=False):
        with torch.inference_mode():
            if not soft:
                hist_ = torch.sparse_coo_tensor(
                    self.labels_.reshape(1, -1), self.act_.flatten(),
                    (self.data.shape[0],), device=self.device,
                ).coalesce().to_dense()
                hist_ = hist_ / (hist_.sum() + _TINY)
                bits_ = self.max_bits + torch.sum(hist_ * torch.log(hist_ + _TINY) / _LOG2)
                return hist_, bits_, None

            dists_ = torch.sum(
                (self.data_.view(-1, 1, 1, self.dim) -
                 self.W_.view(1, self.x, self.y, self.dim)) ** 2,
                dim=3,
            )
            temp_        = torch.tensor(self.median_dist if T is None else T, dtype=FTYPE, device=self.device)
            soft_labels_ = torch.softmax(-dists_ / (temp_ + _TINY), dim=0)
            hist_soft_   = torch.sum(soft_labels_ * self.act_.unsqueeze(0), dim=(1, 2))
            hist_soft_   = hist_soft_ / (hist_soft_.sum() + _TINY)
            bits_soft_   = self.max_bits + torch.sum(hist_soft_ * torch.log(hist_soft_ + _TINY) / _LOG2)
            fingerprint_ = self.act_.flatten() / (self.act_.sum() + _TINY)
            return hist_soft_, bits_soft_, fingerprint_


    def train(
        self, input, alpha=0.1, delta_threshold=0.001, max_steps=100,
        decay=0.25, save_every=None, beta=0.5, in_sigma=1.5, T=1.0,
    ):
        with torch.inference_mode():
            acts    = []
            input   = self.get_input_tensor(input)
            bmu_x, bmu_y = self.present(input=input, T=T)
            dact_   = 1.0
            step    = 0
            while dact_ > delta_threshold and step < max_steps:
                if save_every is not None and step % save_every == 0:
                    acts.append(self.act_.detach().cpu().numpy().copy())
                dact_ = self.next_act(decay=decay)
                step += 1

            # BMU neighborhood
            xs   = torch.arange(self.x, device=self.device, dtype=FTYPE).view(self.x, 1)
            ys   = torch.arange(self.y, device=self.device, dtype=FTYPE).view(1, self.y)
            sig_ = torch.tensor(float(in_sigma), device=self.device, dtype=FTYPE)
            d2_  = (xs - bmu_x.to(FTYPE)).pow(2) + (ys - bmu_y.to(FTYPE)).pow(2)
            N_in = torch.exp(d2_ / (-2.0 * sig_ * sig_ + _TINY))
            N_in = N_in / (N_in.sum() + _TINY)

            N_ex = self.act_ / (self.act_.sum() + _TINY)

            mask         = torch.pow(N_in, 1.0 - beta) * torch.pow(N_ex, beta)
            mask         = mask / (mask.sum() + _TINY)
            alpha_theta_ = mask * (alpha / (mask.max() + _TINY))
            delta_       = alpha_theta_.reshape(self.x, self.y, 1) * (
                input.reshape(1, 1, self.dim) - self.W_
            )
            self.W_.add_(delta_)
            return step, dact_, acts

    def set_lateral(self, sigma=None, lat_strength=0.8, normalize="in", exp=0.5):
        if sigma is None:
            sigma = max(self.x, self.y) / 4.0

        with torch.inference_mode():
            sigma_ = torch.tensor(float(sigma), dtype=FTYPE, device=self.device)
            lat_ex_ = _compiled_set_lateral(
                self.x, self.y, sigma_, lat_strength,
                1.0 / (self.x * self.y), self.device, FTYPE,
            )
            self.lat_ex_ = normalize_lateral_weights(lat_ex_, how=normalize, exp=exp)
            self.lat_in_ = torch.full(
                (self.x, self.y, self.x, self.y),
                1.0 / (self.x * self.y),
                dtype=FTYPE, device=self.device,
            )
            self.lat_ = lat_strength * (self.lat_ex_ - self.lat_in_)
        return self.lat_


def train_som(
    x, y, inputs,
    freq=None, isolation=None, isolation_weight=0.3,
    epochs=1000, alpha=None, sigma=None,
    steps=100, decay=0.2, lat_strength=0.9, thresh=0.001,
    beta=0.5, in_sigma=None, T=1.0,
    n_samples_per_epoch=None, seed_init=True, init_noise_scale=0.01,
    correct_every=10, boost_factor=3.0, boost_passes=5,
    boost_zero_threshold=0, boost_ratio_threshold=0.5,
    ceiling_factor=2.0, verbose_correct=True,
    random_state=None,
):
    if alpha is None:
        alpha = make_decay(initial=0.1, halflife=250)
    if sigma is None:
        sigma = make_decay(initial=1.8, halflife=200)

    if in_sigma is None:
        _in_sigma_fn = make_decay(initial=2.25, halflife=200)
    elif callable(in_sigma):
        _in_sigma_fn = in_sigma
    else:
        _in_sigma_fn = lambda epoch: float(in_sigma)

    if freq is None:
        freq = np.full(len(inputs), 1.0 / len(inputs), dtype=np.float64)
    else:
        freq = np.asarray(freq, dtype=np.float64) / np.asarray(freq, dtype=np.float64).sum()

    if isolation is None:
        print("Computing isolation scores...", flush=True)
        isolation = compute_isolation(inputs)
    isolation = np.asarray(isolation, dtype=np.float64)

    sampling_probs = blend_sampling_weights(freq, isolation, isolation_weight)
    expected_cells = compute_expected_cells(freq, x, y)

    if n_samples_per_epoch is None:
        n_samples_per_epoch = len(inputs)

    rng = np.random.default_rng(random_state)

    if seed_init:
        W0  = make_seeded_weights(data=inputs, x=x, y=y,
                                  noise_scale=init_noise_scale, rng=rng)
        som = LatSom(inputs, W=W0)
    else:
        som = LatSom(inputs, x=x, y=y)

    print("_" * 80)
    progress_every = max(1, epochs // 100)

    for epoch in range(epochs):
        if epoch % progress_every == 0:
            print(".", end="", flush=True)

        current_sigma    = sigma(epoch)
        current_alpha    = alpha(epoch)
        current_in_sigma = _in_sigma_fn(epoch)

        som.set_lateral(sigma=current_sigma, lat_strength=lat_strength,
                        normalize="in", exp=0.5)

        sampled = rng.choice(np.arange(len(inputs)), size=n_samples_per_epoch,
                             replace=True, p=sampling_probs)
        for p in sampled:
            som.train(input=p, alpha=current_alpha, delta_threshold=thresh,
                      max_steps=steps, decay=decay, beta=beta,
                      in_sigma=current_in_sigma, T=T)

        # Territory correction: ceiling first, then boost
        if correct_every and ((epoch + 1) % correct_every == 0):
            under_indices, deficit_ratios, counts = som.get_underrepresented(
                expected_cells=expected_cells,
                zero_threshold=boost_zero_threshold,
                ratio_threshold=boost_ratio_threshold,
            )
            over_indices, excess_ratios = som.get_overrepresented(
                expected_cells=expected_cells, counts=counts,
                ceiling_factor=ceiling_factor,
            )

            if len(over_indices) > 0:
                som.penalise_overrepresented(
                    over_indices=over_indices, excess_ratios=excess_ratios,
                    expected_cells=expected_cells, counts=counts,
                    alpha=current_alpha, sigma=current_sigma,
                    thresh=thresh, steps=steps, decay=decay,
                    beta=beta, in_sigma=current_in_sigma, T=T,
                )

            if len(under_indices) > 0:
                for idx, deficit in zip(under_indices, deficit_ratios):
                    boosted_alpha = current_alpha * (1.0 + boost_factor * float(deficit))
                    for _ in range(boost_passes):
                        som.train(input=int(idx), alpha=boosted_alpha,
                                  delta_threshold=thresh, max_steps=steps,
                                  decay=decay, beta=beta,
                                  in_sigma=current_in_sigma, T=T)

            if verbose_correct:
                counts_after = som.territory_sizes().cpu().numpy()
                n_repr = int((counts_after > 0).sum())
                print(
                    f"\n[epoch {epoch+1}] "
                    f"penalised={len(over_indices)}, boosted={len(under_indices)}, "
                    f"represented={n_repr}/{len(inputs)}",
                    end="", flush=True,
                )

    print()
    counts_final = som.territory_sizes().cpu().numpy()
    n_repr = int((counts_final > 0).sum())
    print(f"Final represented inputs: {n_repr}/{len(inputs)}")
    return som


# ── CELL 6 · Run training ─────────────────────────────────────────────────────

map_size = 90
epochs   = 500
ppe      = 850   # patterns per epoch

# Prepare map data
# Drop freq columns to get the phonetic or semantic features
spa_clean = spa_data.drop(columns=["word_freq", "normalized_word_freq"])
#sem_clean = sem_data.drop(columns=["word_freq", "normalized_word_freq"])
#eng_clean = eng_data.drop(columns=["word_freq", "normalized_word_freq"])

# Probs: frequency probabilities for frequency-based input sampling
probs = spa_data['normalized_word_freq']
probs = probs/probs.sum()

isolation = compute_isolation(spa_clean)

print(f"spa_clean: {spa_clean.shape}, probs: {len(probs)}, isolation: {len(isolation)}")

print(f"Training {map_size}x{map_size} SOM for {epochs} epochs "
      f"({ppe} patterns/epoch) on {DEVICE}...")

t0 = time.time()

track = pd.DataFrame(
    columns=["t", "dt", "n_represented", "sigma", "alpha"],
    index=range(epochs),
)

#SOM training call

# spafreq_som = train_som(
#     x=map_size, y=map_size,
#     inputs=spa_clean,
#     freq=probs,
#     isolation=isolation,
#     isolation_weight=0.3,
#     epochs=epochs,
#     steps=100,
#     decay=0.2,
#     lat_strength=0.9,
#     thresh=0.001,
#     T=0.2,
#     n_samples_per_epoch=ppe,
#     correct_every=10,
#     boost_factor=3.0,
#     boost_passes=5,
#     ceiling_factor=2.0,
#     verbose_correct=True,
#     seed_init=True,
#     random_state=42,
# )

# elapsed = (time.time() - t0) / 60.0
# print(f"\nTraining complete in {elapsed:.2f} minutes")

Using GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
spa_clean: (789, 192), probs: 789, isolation: 789
Training 90x90 SOM for 500 epochs (850 patterns/epoch) on cuda...


Associative Connection Training

In [ ]:
# ---------------------------------------------------------------------------
# 1. Load word frequency from stimuli master and align to the shared vocab
# ---------------------------------------------------------------------------
stimuli = pd.read_csv(
    "stimuli_master_with_coca_2026-04-21.csv", index_col=0   # index = eng word
)

raw_freq = stimuli["eng_freq"]

shared_index = sem_data.index          # words present in the eng map

typ = pd.read_csv("Typicality_outputs_gpt-4o_2025-08-10.csv")
typ.columns = ["word", "category", "US", "Hispanic"]
typ = typ.dropna(subset=["word", "category"])

# Merge English word frequencies
typ = typ.merge(
    eng_data[["word_freq"]],
    left_on="word",
    right_index=True,
    how="left"
).rename(columns={"word_freq": "eng_word_freq"})

# Merge Spanish word frequencies
typ = typ.merge(
    spa_data[["word_freq"]],
    left_on="word",
    right_index=True,
    how="left"
).rename(columns={"word_freq": "spa_word_freq"})

sem_vocab = set(sem_data.index)
eng_vocab = list(eng_data.index)

sem_substitutes = {}

for word in eng_vocab:
    if word in sem_vocab:
        continue

    members = typ[typ["category"] == word]
    if members.empty:
        continue

    in_vocab = [(r["word"], float(r["eng_word_freq"]))
                for _, r in members.iterrows()
                if r["word"] in sem_vocab]
    if not in_vocab:
        continue

    sem_substitutes[word] = in_vocab

print(f"{len(sem_substitutes)} eng-only category words mapped to sem members")


# Re-set lateral weights on both maps with the correct strength
engfreq_som.set_lateral(sigma=0.3, lat_strength=0.9)
semfreq_som.set_lateral(sigma=0.3, lat_strength=0.9)

engfreq_som.update_labels()
semfreq_som.update_labels()

eng_T          = 0.1
sem_T          = 0.1    # keep original sem T
freq_T         = 1.0    # softmax category frequencies
decay          = 0.9
target_bits    = 6.0
tmax           = 300

def _settle_and_accumulate(eng_word, sem_word, accumulator, scale=1.0):
    engfreq_som.present(input=eng_word, T=eng_T)
    semfreq_som.present(input=sem_word, T=sem_T)

    _, engb, _ = engfreq_som.class_hist(soft=False)
    _, semb, _ = semfreq_som.class_hist(soft=False)

    t = 0
    while t <= tmax:
        if engb.item() <= target_bits:
            engfreq_som.next_act(decay=decay)
            _, engb, _ = engfreq_som.class_hist(soft=False)
        if semb.item() <= target_bits:
            semfreq_som.next_act(decay=decay)
            _, semb, _ = semfreq_som.class_hist(soft=False)
        if engb.item() > target_bits and semb.item() > target_bits:
            break
        t += 1

    co_act_ = (
        engfreq_som.act_.reshape(engfreq_som.x, engfreq_som.y, 1, 1)
        * semfreq_som.act_.reshape(1, 1, semfreq_som.x, semfreq_som.y)
    )
    if scale != 1.0:
        co_act_ = co_act_ * scale
    torch.add(accumulator, co_act_, out=accumulator)


with torch.inference_mode():
    coact_W_ = torch.zeros(
        [engfreq_som.x, engfreq_som.y, semfreq_som.x, semfreq_som.y],
        dtype=FTYPE, device=DEVICE
    )

    for epoch in range(10):
        print(f"\nepoch {epoch}", flush=True)

        for word_idx, word in enumerate(eng_vocab):
            if word_idx % 20 == 0:
                print(word, flush=True)

            if word in sem_vocab:
                # direct case: eng word is also a sem-vocab member
                _settle_and_accumulate(word, word, coact_W_)

            elif word in sem_substitutes:
                members = sem_substitutes[word]
                subordinate = [m[0] for m in members]
                freqs = np.array([m[1] for m in members], dtype=float)
                freq_T = np.log(len(freqs) + 1)
                freq_weights = np.exp((freqs - freqs.max()) / freq_T)
                freq_weights /= freq_weights.sum()

                for member, scale in zip(subordinate, freq_weights):
                    _settle_and_accumulate(word, member, coact_W_, scale)
            else:
                continue

    eng_sum_ = torch.sum(coact_W_, dim=[0, 1], keepdim=True)
    eng2sem_W_ = coact_W_ / eng_sum_.clamp(min=1e-10)

    sem_sum_ = torch.sum(coact_W_, dim=[2, 3], keepdim=True)
    sem2eng_W_ = coact_W_ / sem_sum_.clamp(min=1e-10)

print("eng2sem training complete. Shape:", eng2sem_W_.shape)
print("sem2eng training complete. Shape:", sem2eng_W_.shape)

121 eng-only category words mapped to sem members

epoch 0
acai
baby
bed
bottle
bus
cardinal
church
corn
desert
duck
finch
fountain
grapes
heel
jacket
lark
llama
monkey
nurse
papaya
picture
pomelo
raft
rocket
scale
shoulder
sock
stoplight
tail
toaster
turban
vest
wheel
zipper
child
farm animal
insect
musical instrument
seed
vegetable

epoch 1
acai
baby
bed
bottle
bus
cardinal
church
corn
desert
duck
finch
fountain
grapes
heel
jacket
lark
llama
monkey
nurse
papaya
picture
pomelo
raft
rocket
scale
shoulder
sock
stoplight
tail
toaster
turban
vest
wheel
zipper
child
farm animal
insect
musical instrument
seed
vegetable

epoch 2
acai
baby
bed
bottle
bus
cardinal
church
corn
desert
duck
finch
fountain
grapes
heel
jacket
lark
llama
monkey
nurse
papaya
picture
pomelo
raft
rocket
scale
shoulder
sock
stoplight
tail
toaster
turban
vest
wheel
zipper
child
farm animal
insect
musical instrument
seed
vegetable

epoch 3
acai
baby
bed
bottle
bus
cardinal
church
corn
desert
duck
finch
fountain
grapes
hee

In [ ]:
Naming Simulations

In [ ]:
#English and/or Spanish naming simulations

def sem_to_readout(
    word,
    sem_T=0.1, eng_T=0.1, spa_T=0.1,
    decay=0.9,
    g_assoc=0.1,
    g_decay=20.0,
    target_bits=7.0,
    tmax=1000,
    lang="eng",            # lang="spa" for Spanish naming
    nontargetlang="spa",   # choose the other language
):
    soms = {"eng": engfreq_som, "spa": spafreq_som, "sem": semfreq_som}

    with torch.inference_mode():
        for som in (semfreq_som, engfreq_som, spafreq_som):
            if som.lat_ is None:
                som.set_lateral(sigma=0.3, lat_strength=0.9)

        # Present naming item to the semantic map
        semfreq_som.present(input=word, T=sem_T)
        engfreq_som.act_.zero_()
        spafreq_som.act_.zero_()

        t = 0
        target_bits_val = float("inf")
        while t < tmax:

            g_t = g_assoc * float(np.exp(-t / g_decay))

            sem_act = semfreq_som.act_.clone()
            eng_act = engfreq_som.act_.clone()
            spa_act = spafreq_som.act_.clone()

            sem_drive = (
                torch.einsum("ij,ijkl->kl", eng_act, eng2sem_W_) +
                torch.einsum("ij,ijkl->kl", spa_act, spa2sem_W_)
            ) * g_t

            eng_drive = (
                torch.einsum("kl,ijkl->ij", sem_act, sem2eng_W_) +
                torch.einsum("kl,ijkl->ij", spa_act, spa2eng_W_)
            ) * g_t

            spa_drive = (
                torch.einsum("kl,ijkl->ij", sem_act, sem2spa_W_) +
                torch.einsum("ij,ijkl->kl", eng_act, eng2spa_W_)
            ) * g_t

            semfreq_som.next_act(decay=decay, other_input=sem_drive)
            engfreq_som.next_act(decay=decay, other_input=eng_drive)
            spafreq_som.next_act(decay=decay, other_input=spa_drive)

            # Measure target lang map certainty
            _, target_b, _ = soms[lang].class_hist(soft=False)
            target_bits_val = target_b.item()

            if target_bits_val >= target_bits:
                break

            t += 1

        eng_words = engfreq_som.data.index
        spa_words = spafreq_som.data.index
        sem_words = semfreq_som.data.index
        words = {"eng": eng_words, "spa": spa_words, "sem": sem_words}

        eng_hist, _, _ = engfreq_som.class_hist(soft=False)
        spa_hist, _, _ = spafreq_som.class_hist(soft=False)
        sem_hist, _, _ = semfreq_som.class_hist(soft=False)
        hists = {"eng": eng_hist, "spa": spa_hist, "sem": sem_hist}

        target_word_idx = int(hists[lang].argmax().item())
        nontarget_word_idx = int(hists[nontargetlang].argmax().item())
        sem_word_idx = int(sem_hist.argmax().item())

        return {
            "target_word": words[lang][target_word_idx],
            "target_bits": target_bits_val,
            "steps": t,
            "nontargetlang_word": words[nontargetlang][nontarget_word_idx],
            "sem_word": sem_words[sem_word_idx],
        }